# Hostile Testing Phase 8 - Testing Framework

## Purpose
Test testing utilities and runner functions with hostile inputs

## Goal
Add 6+ functions, reach 7.5%+ coverage

In [ ]:
import pandas as pd
import tempfile
from pathlib import Path

test_results = {
    'function': [],
    'module': [],
    'test_category': [],
    'passed': [],
    'error_message': [],
    'severity': []
}

def record_test(function, module, test_category, passed, error_message="", severity="medium"):
    test_results['function'].append(function)
    test_results['module'].append(module)
    test_results['test_category'].append(test_category)
    test_results['passed'].append(passed)
    test_results['error_message'].append(error_message)
    test_results['severity'].append(severity)

def hostile_test(func, test_name, *args, **kwargs):
    try:
        result = func(*args, **kwargs)
        return (True, result, "")
    except Exception as e:
        return (False, None, str(e))

print("Phase 8 test harness loaded - TESTING FRAMEWORK")

## Section 1: Testing Runner - Test Execution

In [ ]:
from siege_utilities import (
    discover_tests, run_test_suite, generate_test_report
)

# Test discover_tests with hostile paths
hostile_test_paths = [
    None,
    "",
    "'; DROP TABLE tests; --",
    "<script>alert(1)</script>",
    "../../../etc/passwd",
    "/etc/shadow",
    "test; rm -rf /",
    "$(cat /etc/passwd)",
    "`whoami`",
    "A" * 10000,
    "path\x00null",
    "/nonexistent/path/to/tests",
]

for path in hostile_test_paths:
    success, result, error = hostile_test(discover_tests, "hostile_path", path)
    record_test(
        "discover_tests",
        "testing.runner",
        "hostile_paths",
        success,
        error,
        "critical" if "DROP TABLE" in str(path) or "rm -rf" in str(path) or "$(" in str(path) or "`" in str(path) else "high" if "../" in str(path) or "/etc/" in str(path) else "medium"
    )

print(f"Completed {len([r for r in test_results['function'] if r == 'discover_tests'])} discover_tests tests")

## Section 2: Testing Environment - Setup & Teardown

In [ ]:
from siege_utilities import (
    create_test_environment, cleanup_test_environment, get_test_data_dir
)

# Test create_test_environment with hostile configs
hostile_configs = [
    {"name": "'; DROP TABLE envs; --", "temp_dir": "/tmp"},
    {"name": "<script>alert(1)</script>", "temp_dir": "../../../etc"},
    {"name": "env; rm -rf /", "temp_dir": "/etc/passwd"},
    {"name": "A" * 10000, "temp_dir": "B" * 10000},
    {"name": "env\x00null", "temp_dir": "dir\x00null"},
    {"name": None, "temp_dir": None},
    {"name": "", "temp_dir": ""},
    {"name": "test_env", "temp_dir": "/tmp"},  # Valid
]

for cfg in hostile_configs:
    success, result, error = hostile_test(
        create_test_environment, "hostile_env",
        cfg.get('name'), cfg.get('temp_dir')
    )
    record_test(
        "create_test_environment",
        "testing.environment",
        "hostile_configs",
        success,
        error,
        "critical" if "DROP TABLE" in str(cfg.get('name','')) or "rm -rf" in str(cfg.get('name','')) else "high" if "../" in str(cfg.get('temp_dir','')) or "/etc/" in str(cfg.get('temp_dir','')) else "medium"
    )

# Test get_test_data_dir (no args)
success, result, error = hostile_test(get_test_data_dir, "get_data_dir")
record_test("get_test_data_dir", "testing.environment", "basic_call", success, error, "low")

print(f"Completed {len([r for r in test_results['module'] if 'testing.environment' in r])} environment tests")

## Section 3: Data Sample - Dataset Access

In [ ]:
from siege_utilities import load_sample_data, get_sample_data_path

# Test load_sample_data with hostile dataset names
hostile_datasets = [
    None,
    "",
    "'; DROP TABLE datasets; --",
    "<script>alert(1)</script>",
    "../../../etc/passwd",
    "dataset; rm -rf /",
    "$(cat /etc/passwd)",
    "`whoami`",
    "A" * 10000,
    "dataset\x00null",
    "数据集",  # Unicode
    "iris",  # Valid (if exists)
]

for dataset in hostile_datasets:
    success, result, error = hostile_test(load_sample_data, "hostile_dataset", dataset)
    record_test(
        "load_sample_data",
        "data.sample_data",
        "hostile_datasets",
        success,
        error,
        "critical" if "DROP TABLE" in str(dataset) or "rm -rf" in str(dataset) or "$(" in str(dataset) or "`" in str(dataset) else "high" if "../" in str(dataset) else "medium"
    )

# Test get_sample_data_path with hostile dataset names
for dataset in hostile_datasets[:8]:  # First 8 only
    success, result, error = hostile_test(get_sample_data_path, "hostile_dataset", dataset)
    record_test(
        "get_sample_data_path",
        "data.sample_data",
        "hostile_datasets",
        success,
        error,
        "high" if "../" in str(dataset) or "DROP" in str(dataset) else "medium"
    )

print(f"Completed {len([r for r in test_results['module'] if 'sample_data' in r])} sample data tests")

## Section 4: Generate Phase 8 Results

In [ ]:
df = pd.DataFrame(test_results)

print("\n" + "="*80)
print("PHASE 8 HOSTILE TESTING SUMMARY")
print("="*80)
print(f"\nTests run: {len(df)}")
print(f"Passed: {df['passed'].sum()}")
print(f"Failed: {(~df['passed']).sum()}")
print(f"Pass rate: {df['passed'].sum() / len(df) * 100:.1f}%")

unique = df['function'].nunique()
print(f"\nUnique functions: {unique}")
print(f"Phase 8 coverage: {unique}/751 = {unique/751*100:.1f}%")

print("\n" + "="*80)
print("RESULTS BY MODULE")
print("="*80)
summary = df.groupby('module').agg({'passed': ['sum', 'count']})
summary.columns = ['passed', 'total']
summary['pass_rate'] = (summary['passed'] / summary['total'] * 100).round(1)
print(summary)

failures = df[~df['passed']]
if len(failures) > 0:
    print("\n" + "="*80)
    print("FAILURES BY SEVERITY")
    print("="*80)
    print(failures.groupby('severity').size())
    
    critical_high = failures[failures['severity'].isin(['critical', 'high'])]
    if len(critical_high) > 0:
        print("\n" + "="*80)
        print("CRITICAL/HIGH FAILURES (First 5)")
        print("="*80)
        for idx, row in critical_high.head(5).iterrows():
            print(f"\n{row['function']} ({row['severity']})")
            print(f"  Module: {row['module']}")
            print(f"  Error: {row['error_message'][:100]}")
else:
    print("\n✅ NO FAILURES")

df.to_csv("hostile_testing_phase8_results.csv", index=False)
print(f"\nSaved: hostile_testing_phase8_results.csv")

print("\n" + "="*80)
print("FUNCTIONS TESTED")
print("="*80)
for func in df['function'].unique():
    tests = df[df['function'] == func]
    passed = tests['passed'].sum()
    total = len(tests)
    print(f"{func}: {passed}/{total} ({passed/total*100:.0f}%)")

print("\n" + "="*80)
print("CUMULATIVE COVERAGE (Phases 1-8)")
print("="*80)
total_unique = 51 + unique
print(f"Total unique functions: ~{total_unique}")
print(f"Cumulative coverage: ~{total_unique}/751 = {total_unique/751*100:.1f}%")
print(f"\n🎯 Progress to 25% goal: {total_unique}/188 = {total_unique/188*100:.1f}%")